# Scoring d'impayé 

In [1]:
import pandas as pd
import numpy as np
import feature_engineering as fe
import eda_viz as viz

In [2]:
df = pd.read_csv('data/data.csv')
df['subscription_datetime'] = pd.to_datetime(df['subscription_datetime'])
df = df.drop_duplicates().reset_index(drop=True)
df = df[df['monthly_amount'] != 0].reset_index(drop=True)

# Déclaration centrale (cohérente avec eda.ipynb)
TARGET = 'top_unpaid'
DATE_COL = 'subscription_datetime'
NUM_COLS = ['monthly_amount']
GEO_COL = 'zip_code_prefix'
CAT_COLS = ['channel', 'energy_type', 'phone_carrier', 'bank_name',
            'email_domain', 'ported_phone_number']

print(f'Dataset : {len(df):,} lignes  |  taux global : {df[TARGET].mean()*100:.2f}%')

Dataset : 4,933 lignes  |  taux global : 30.18%


In [3]:
cols_to_check = ['ported_phone_number', 'energy_type', 'channel']

mask_multi_missing = df[cols_to_check].isna().sum(axis=1) >= 2

df = df[~mask_multi_missing]

## Phase I: Feature Engineering

Cette partie applique les 5 phases du pipeline en s'appuyant sur les
transformations définies dans `feature_engineering.py`. Principes :

- **Tout est fitté sur le train**, appliqué tel quel à val/test.
- **Le split est temporel**, pas aléatoire, on entraîne sur le passé,
  on évalue sur le futur, comme en production.

#### a. Split temporel 70%/15%/15%

La manière idéale de partitionner les données est d'effectuer un split temporel,
afin de simuler les conditions de production et d'évaluer le time-drift du modèle.
Dans un contexte de scoring d'acquistion, cela signifie constituer trois cohortes
distinctes (T0, T1, T2) avec une fenêtre d'observation identique pour le
label (ex. impayé à 6 mois), garantissant que le modèle est entraîné et évalué
sur la même définition du risque.

In [4]:
train, val, test = fe.temporal_split(df, DATE_COL, train_frac=0.70, val_frac=0.15)
fe.split_summary(train, val, test, TARGET, DATE_COL)

,split,n,période_début,période_fin,taux_impayé_%
0,train,3304,2024-04-29,2024-11-16,39.26
1,val,708,2024-11-16,2025-01-02,6.07
2,test,708,2025-01-03,2025-03-14,11.72


**Biais de maturité :** ici, `top_unpaid` est cumulatif depuis la date de souscription
jusqu'à la date d'extraction, et il n'existe qu'une seule photo dans le temps.
Les clients du train ont donc une fenêtre d'observation bien plus longue que ceux
du val/test, ce qui surestime mécaniquement le taux d'impayé en train (39 % vs 6–12 %).
Ce biais est documenté et accepté compte tenu des contraintes du jeu de données.

#### b. Valeurs manquantes

Imputation simple :
- `"MISSING"` pour les catégorielles
- médiane (figée sur train) pour les numériques

In [5]:
mh = fe.MissingHandler(cat_cols=CAT_COLS, num_cols=NUM_COLS).fit(train)
train = mh.transform(train)
val   = mh.transform(val)
test  = mh.transform(test)

print('Médianes figées (train) :', mh.medians_)

Médianes figées (train) : {'monthly_amount': 7.0}


#### c. Outliers

In [6]:
# Troncature de monthly_amount au 99e centile (winsorisation)
# Le plafond est calculé sur train, puis réappliqué tel quel à val/test
train, cap_monthly = fe.truncate_at_percentile(train, 'monthly_amount', percentile=99)
val,  _ = fe.truncate_at_percentile(val,  'monthly_amount', cap=cap_monthly)
test, _ = fe.truncate_at_percentile(test, 'monthly_amount', cap=cap_monthly)

print(f'Plafond (99e centile train) : {cap_monthly}')
print('Distribution monthly_amount (train) après troncature :')
print(train['monthly_amount'].describe().round(2))

Plafond (99e centile train) : 24.0
Distribution monthly_amount (train) après troncature :
count    3304.00
mean        8.23
std         4.50
min         1.00
25%         5.00
50%         7.00
75%        11.00
max        24.00
Name: monthly_amount, dtype: float64


#### d. Features dérivées

**Regroupements sémantiques** à partir de variables existantes :
- *Densité urbaine* (`urban_density`) : chaque code postal est converti en catégorie de densité (métropole dense, urbain, périurbain, rural…) via le dataset INSEE.
- *Banques* (`bank_name`) : les modalités sont agrégées en groupes cohérents via `BANK_GROUPS`.
- *E-mail* (`email_tld`, `email_is_french`, `email_is_webmail`, `email_domain_length`) : `email_domain` est remplacée par des features plus informatives.


In [7]:
# 1. Construction de la table de densité (une fois)
DENSITY_TABLE = fe.build_dept_density_table()
print(f"Table de densité : {len(DENSITY_TABLE)} départements couverts")

# 2. Appliquer à chaque split
train = fe.encode_zip_urban_density(train, DENSITY_TABLE)
val   = fe.encode_zip_urban_density(val,   DENSITY_TABLE)
test  = fe.encode_zip_urban_density(test,  DENSITY_TABLE)

# 3. Vérifier
print("\nDistribution train :")
print(train['urban_density'].value_counts(normalize=True).mul(100).round(1))

# 4. Vérifier qu'il n'y a pas trop de 'unknown'
unknown_pct = (train['urban_density'] == 'unknown').mean() * 100
print(f"\n% unknown en train : {unknown_pct:.2f}%")

Table de densité : 95 départements couverts

Distribution train :
urban_density
périurbain         41.6
urbain             35.7
rural               8.1
urbain_dense        7.4
métropole_dense     7.0
unknown             0.1
Name: proportion, dtype: float64

% unknown en train : 0.09%


In [8]:
# Regroupement sémantique de bank_name
MANUAL_GROUPS = {'bank_name': fe.BANK_GROUPS}
train = fe.apply_manual_groups(train, MANUAL_GROUPS)
val   = fe.apply_manual_groups(val,   MANUAL_GROUPS)
test  = fe.apply_manual_groups(test,  MANUAL_GROUPS)

# Features dérivées de email_domain
train = fe.derive_email_features(train)
val   = fe.derive_email_features(val)
test  = fe.derive_email_features(test)

CAT_COLS = [c for c in CAT_COLS ] + ['email_tld', 'email_is_french', 'email_is_webmail']
EMAIL_NUM_FEATURES = ['email_domain_length']

print("Nouvelle distribution bank_name :")
print(train['bank_name'].value_counts(normalize=True).mul(100).round(1))
print("\nFeatures email dérivées :", ['email_tld', 'email_is_french', 'email_is_webmail'] + EMAIL_NUM_FEATURES)

Nouvelle distribution bank_name :
bank_name
banque_traditionnelle    43.8
OTHER                    31.1
banque_postale            9.7
neobanque                 8.0
banque_en_ligne           7.4
Name: proportion, dtype: float64

Features email dérivées : ['email_tld', 'email_is_french', 'email_is_webmail', 'email_domain_length']


#### e. Regroupement des modalités rares

Toute modalité représentant **moins de 1% du volume en train** est remplacée
par `"OTHER"`. Le seuil est calculé uniquement sur train ; les modalités
nouvelles en val/test (jamais vues en train) tombent aussi dans `OTHER`.

In [9]:
rg = fe.RareCategoryGrouper(cols=CAT_COLS, min_freq=0.01).fit(train)
train = rg.transform(train)
val   = rg.transform(val)
test  = rg.transform(test)

rg.summary()

,variable,n_modalités_gardées,modalités_gardées
0,channel,3,"[MISSING, PHONE, WEB]"
1,energy_type,3,"[ELEC, GAZ, MISSING]"
2,phone_carrier,6,"[BOUYGUES TELECOM, FREE MOBILE, LYCAMOBILE, MI..."
3,bank_name,5,"[OTHER, banque_en_ligne, banque_postale, banqu..."
4,email_domain,12,"[FREE.FR, GMAIL.COM, HOTMAIL.COM, HOTMAIL.FR, ..."
5,ported_phone_number,3,"[0.0, 1.0, MISSING]"
6,email_tld,4,"[com, fr, net, unknown]"
7,email_is_french,2,"[0, 1]"
8,email_is_webmail,2,"[0, 1]"


#### f. Vérifications finales

Les features sont séparées en deux groupes pour l'analyse de colinéarité :

- **Numériques** (`FINAL_NUM_COLS`) : corrélation de Spearman, robuste aux distributions asymétriques.
- **Catégorielles** (`FINAL_CAT_COLS`) : V de Cramér sur les colonnes originales post-regroupement, identique à l'EDA.

In [10]:
# Features numériques : tout ce qui n'est ni catégoriel brut, ni ID, ni cible, ni date
EXCLUDED = {TARGET, DATE_COL, GEO_COL}  # target, date, et le zip brut (déjà encodé)

NUMERIC_FEATURES = [
    c for c in train.columns
    if c not in EXCLUDED
    and pd.api.types.is_numeric_dtype(train[c])
]

CATEGORICAL_FEATURES = [
    c for c in train.columns
    if c not in EXCLUDED
    and not pd.api.types.is_numeric_dtype(train[c])

]

print(f"Numériques ({len(NUMERIC_FEATURES)}) :")
for f in NUMERIC_FEATURES:
    print(f"  - {f}")

print(f"\nCatégorielles ({len(CATEGORICAL_FEATURES)}) :")
for f in CATEGORICAL_FEATURES:
    print(f"  - {f}")

Numériques (4) :
  - monthly_amount
  - email_is_french
  - email_is_webmail
  - email_domain_length

Catégorielles (8) :
  - email_domain
  - phone_carrier
  - ported_phone_number
  - channel
  - energy_type
  - bank_name
  - urban_density
  - email_tld


In [11]:
viz.plot_numeric_correlation(train, NUMERIC_FEATURES, method = "spearman").show()

In [12]:
viz.plot_cramers_matrix(train, CATEGORICAL_FEATURES).show()

In [13]:
# Suppression de email_tld : retirée des features et des DataFrames (variable trop correlée).
DROP_COL = 'email_tld'

CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if c != DROP_COL]
CAT_COLS = [c for c in CAT_COLS if c != DROP_COL]

train = train.drop(columns=DROP_COL, errors='ignore')
val   = val.drop(columns=DROP_COL,   errors='ignore')
test  = test.drop(columns=DROP_COL,  errors='ignore')

print(f"'{DROP_COL}' supprimée.")
print(f"Catégorielles ({len(CATEGORICAL_FEATURES)}) :", CATEGORICAL_FEATURES)

'email_tld' supprimée.
Catégorielles (7) : ['email_domain', 'phone_carrier', 'ported_phone_number', 'channel', 'energy_type', 'bank_name', 'urban_density']


In [14]:
# Ranking de toutes les variables
all_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
viz.iv_summary(train, all_cols, TARGET)

,variable,IV,pouvoir_predictif
0,bank_name,0.7565,Suspect
1,phone_carrier,0.1848,Moyen
2,email_domain,0.1832,Moyen
3,ported_phone_number,0.1325,Moyen
4,urban_density,0.1043,Moyen
5,channel,0.0574,Faible
6,email_is_french,0.0532,Faible
7,email_is_webmail,0.0521,Faible
8,monthly_amount,0.0336,Faible
9,energy_type,0.0216,Faible


## Phase II: Modelisation

### Setup - MLflow

Tous les modèles entraînés ci-dessous sont loggués dans la même expérience
MLflow `scoring_impaye` via `modelisation.evaluate_and_log`. Cela permet :

- de garder une trace versionnée des params et métriques ;
- de comparer les 4 expériences (a, b, c, d) à la fin via `mdl.compare_runs()`.

Le seuil métier est toujours calé sur **val** sous contrainte `recall ≥ 0.30`.

In [15]:
import modelisation as mdl

# Crée/récupère l'expérience locale (./mlruns par défaut)
mdl.setup_mlflow("scoring_impaye")

2026/06/30 16:38:03 INFO mlflow.tracking.fluent: Experiment with name 'scoring_impaye' does not exist. Creating a new experiment.


'scoring_impaye'

### a. Baseline - Random Forest

On entraîne un Random Forest sur les features finales. `class_weight='balanced'` compense le déséquilibre de classes. L'objectif est d'obtenir une precision de référence avant toute optimisation.

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_train = train[ALL_FEATURES]
y_train = train[TARGET]
X_val   = val[ALL_FEATURES]
y_val   = val[TARGET]
X_test  = test[ALL_FEATURES]
y_test  = test[TARGET]

cat_pipeline = Pipeline([
    ("to_str", FunctionTransformer(lambda X: X.astype(str))),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", "passthrough", NUMERIC_FEATURES),
    ("cat", cat_pipeline, CATEGORICAL_FEATURES),
], remainder="drop")

rf_pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,        
        min_samples_leaf=10,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

rf_pipeline.fit(X_train, y_train)

ohe_cols = (rf_pipeline.named_steps["prep"]
            .named_transformers_["cat"]
            .named_steps["ohe"]
            .get_feature_names_out(CATEGORICAL_FEATURES).tolist())

ALL_FEATURE_NAMES = NUMERIC_FEATURES + ohe_cols

print("Pipeline Random Forest entraîné")
print(f"  Numériques (passthrough) : {len(NUMERIC_FEATURES)}")
print(f"  Catégorielles (OHE)      : {len(ohe_cols)}")
print(f"  Total features           : {len(ALL_FEATURE_NAMES)}")

Pipeline Random Forest entraîné
  Numériques (passthrough) : 4
  Catégorielles (OHE)      : 40
  Total features           : 44


In [17]:
# 1. Probabilités sur val (et train pour estimer overfitting)
proba_rf_val  = rf_pipeline.predict_proba(X_val)[:, 1]
proba_rf_train = rf_pipeline.predict_proba(X_train)[:, 1]

# 2. Seuil optimal + métriques + courbe PR + log MLflow (en une ligne)
rf_result = mdl.evaluate_and_log(
    model_name="Random Forest",
    run_name="a_random_forest_baseline",
    params={
        "model_type":     "RandomForestClassifier",
        "n_estimators":   300,
        "min_samples_leaf": 10,
        "max_features":   "sqrt",
        "class_weight":   "balanced",
        "encoding":       "OneHotEncoder",
        "n_features":     len(ALL_FEATURE_NAMES),
    },
    y_val=y_val,  proba_val=proba_rf_val,
    y_train=y_train, proba_train=proba_rf_train,
    model=rf_pipeline,
    model_flavor="sklearn",
    tags={"stage": "a_baseline", "encoding": "OHE"},
)

# On garde le seuil pour comparaison éventuelle
SEUIL_OPTIMAL = rf_result["threshold"]

── Random Forest — val ──
  AP (PR-AUC)   : 0.2597
  ROC-AUC       : 0.7436
  Gini          : 0.4873
  Seuil         : 0.639
  Faux Positifs : 35  ← métrique cible
              precision    recall  f1-score   support

        payé      0.955     0.947     0.951       665
      impayé      0.271     0.302     0.286        43

    accuracy                          0.908       708
   macro avg      0.613     0.625     0.618       708
weighted avg      0.913     0.908     0.911       708

── Random Forest — train ──
  AP (PR-AUC)   : 0.7610
  ROC-AUC       : 0.8293
  Gini          : 0.6586
  Seuil         : 0.639
  Faux Positifs : 197  ← métrique cible
              precision    recall  f1-score   support

        payé      0.743     0.902     0.815      2007
      impayé      0.773     0.518     0.620      1297

    accuracy                          0.751      3304
   macro avg      0.758     0.710     0.718      3304
weighted avg      0.755     0.751     0.739      3304



2026/06/30 16:38:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/06/30 16:38:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


### b. LightGBM

LightGBM gère **nativement les variables catégorielles**. pas besoin
de OneHotEncoder. Il suffit de convertir les colonnes en `category`.

Avantages vs RF + OHE :
- 6 colonnes catégorielles brutes au lieu de N modalités one-hot →
  arbres plus efficaces, moins de surapprentissage.
- `is_unbalance=True` compense le déséquilibre de classes.
- Early stopping sur val pour caler `n_estimators` automatiquement.

In [18]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from pandas.api.types import CategoricalDtype

# 1. On fige le vocabulaire des catégorielles SUR TRAIN, puis on l'applique
#    tel quel à val/test (zéro fuite : toute modalité inconnue de val/test
#    sera NaN -> traitée comme une catégorie manquante par LightGBM).

cat_dtypes = {c: CategoricalDtype(categories=X_train[c].unique())
              for c in CATEGORICAL_FEATURES}

X_train_lgb = X_train.astype(cat_dtypes)
X_val_lgb   = X_val.astype(cat_dtypes)
X_test_lgb  = X_test.astype(cat_dtypes)

# 2. Modèle
lgb_model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    feature_fraction=0.9,
    bagging_fraction=0.9,
    bagging_freq=5,
    is_unbalance=True,
    objective="binary",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

lgb_model.fit(
    X_train_lgb, y_train,
    eval_set=[(X_val_lgb, y_val)],
    eval_metric="average_precision",
    callbacks=[early_stopping(50, verbose=False), log_evaluation(0)],
)

print(f"LightGBM entraîné  (best_iteration = {lgb_model.best_iteration_})")

LightGBM entraîné  (best_iteration = 82)


In [19]:
proba_lgb_val  = lgb_model.predict_proba(X_val_lgb)[:, 1]
proba_lgb_train = lgb_model.predict_proba(X_train_lgb)[:, 1]

lgb_result = mdl.evaluate_and_log(
    model_name="LightGBM",
    run_name="b_lightgbm_baseline",
    params={
        "model_type":        "LGBMClassifier",
        "n_estimators_max":  1000,
        "best_iteration":    lgb_model.best_iteration_,
        "learning_rate":     0.05,
        "num_leaves":        31,
        "min_child_samples": 20,
        "feature_fraction":  0.9,
        "bagging_fraction":  0.9,
        "is_unbalance":      True,
        "encoding":          "native_categorical",
    },
    y_val=y_val,  proba_val=proba_lgb_val,
    y_train=y_train, proba_train=proba_lgb_train,
    model=lgb_model,
    model_flavor="lightgbm",
    tags={"stage": "b_lightgbm", "encoding": "native_cat"},
)

── LightGBM — val ──
  AP (PR-AUC)   : 0.3039
  ROC-AUC       : 0.7524
  Gini          : 0.5048
  Seuil         : 0.712
  Faux Positifs : 26  ← métrique cible
              precision    recall  f1-score   support

        payé      0.955     0.961     0.958       665
      impayé      0.333     0.302     0.317        43

    accuracy                          0.921       708
   macro avg      0.644     0.632     0.638       708
weighted avg      0.917     0.921     0.919       708

── LightGBM — train ──
  AP (PR-AUC)   : 0.7967
  ROC-AUC       : 0.8647
  Gini          : 0.7294
  Seuil         : 0.712
  Faux Positifs : 130  ← métrique cible
              precision    recall  f1-score   support

        payé      0.738     0.935     0.825      2007
      impayé      0.829     0.485     0.612      1297

    accuracy                          0.758      3304
   macro avg      0.783     0.710     0.718      3304
weighted avg      0.773     0.758     0.741      3304



2026/06/30 16:38:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/06/30 16:38:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


### c. CatBoost

CatBoost gère lui aussi les catégorielles nativement, mais avec un
**ordered target encoding** interne qui évite les fuites cible. Il est
souvent plus performant que LightGBM sur des datasets tabulaires moyens
avec beaucoup de catégorielles à modalités variées (cas typique du
scoring).

- `auto_class_weights='Balanced'` pour compenser le déséquilibre.
- `eval_metric='PRAUC'` aligne l'optimisation interne sur notre KPI.
- Early stopping sur val.

In [20]:
from catboost import CatBoostClassifier

# CatBoost exige des strings pour les cat_features (pas de NaN). Les
# MissingHandler + RareCategoryGrouper de feature_engineering.py ont
# déjà éliminé les NaN : il reste juste à forcer le dtype string.
X_train_cb = X_train.copy()
X_val_cb   = X_val.copy()
X_test_cb  = X_test.copy()

for c in CATEGORICAL_FEATURES:
    X_train_cb[c] = X_train_cb[c].astype(str)
    X_val_cb[c]   = X_val_cb[c].astype(str)
    X_test_cb[c]  = X_test_cb[c].astype(str)

cb_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    eval_metric="PRAUC",
    auto_class_weights="Balanced",
    random_seed=42,
    early_stopping_rounds=50,
    verbose=False,
)

cb_model.fit(
    X_train_cb, y_train,
    cat_features=CATEGORICAL_FEATURES,
    eval_set=(X_val_cb, y_val),
)

print(f"CatBoost entraîné  (best_iteration = {cb_model.get_best_iteration()})")

CatBoost entraîné  (best_iteration = 114)


In [21]:
proba_cb_val  = cb_model.predict_proba(X_val_cb)[:, 1]
proba_cb_train = cb_model.predict_proba(X_train_cb)[:, 1]

cb_result = mdl.evaluate_and_log(
    model_name="CatBoost",
    run_name="c_catboost_baseline",
    params={
        "model_type":     "CatBoostClassifier",
        "iterations_max": 1000,
        "best_iteration": cb_model.get_best_iteration(),
        "learning_rate":  0.05,
        "depth":          6,
        "l2_leaf_reg":    3.0,
        "auto_class_weights": "Balanced",
        "eval_metric":    "PRAUC",
        "encoding":       "native_categorical",
    },
    y_val=y_val,  proba_val=proba_cb_val,
    y_train=y_train, proba_train=proba_cb_train,
    model=cb_model,
    model_flavor="catboost",
    tags={"stage": "c_catboost", "encoding": "native_cat"},
)

── CatBoost — val ──
  AP (PR-AUC)   : 0.3092
  ROC-AUC       : 0.7494
  Gini          : 0.4988
  Seuil         : 0.692
  Faux Positifs : 29  ← métrique cible
              precision    recall  f1-score   support

        payé      0.958     0.956     0.957       665
      impayé      0.341     0.349     0.345        43

    accuracy                          0.919       708
   macro avg      0.649     0.653     0.651       708
weighted avg      0.920     0.919     0.920       708

── CatBoost — train ──
  AP (PR-AUC)   : 0.7420
  ROC-AUC       : 0.8154
  Gini          : 0.6307
  Seuil         : 0.692
  Faux Positifs : 202  ← métrique cible
              precision    recall  f1-score   support

        payé      0.738     0.899     0.811      2007
      impayé      0.764     0.505     0.608      1297

    accuracy                          0.745      3304
   macro avg      0.751     0.702     0.709      3304
weighted avg      0.748     0.745     0.731      3304



2026/06/30 16:38:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/06/30 16:38:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


### d. Optimisation des hyperparamètres LightGBM

Recherche bayésienne avec **Optuna** (sampler TPE) sur l'AP de val.
Contrairement à GridSearch (exhaustif) ou RandomSearch (aveugle), TPE
apprend des essais précédents pour concentrer l'exploration sur les
régions prometteuses de l'espace, plus efficace à budget d'essais égal.

L'objectif est d'améliorer le modèle `b_lightgbm_baseline` sur le même
search-space, en conservant `early_stopping(50)` à chaque essai pour
éviter le surapprentissage.

Search space :
- `num_leaves` ∈ [20, 200]
- `max_depth` ∈ [3, 12]
- `min_child_samples` ∈ [10, 100]
- `feature_fraction` ∈ [0.5, 1.0]
- `bagging_fraction` ∈ [0.5, 1.0]
- `bagging_freq` ∈ [1, 10]
- `reg_alpha` ∈ [1e-4, 10.0] log-uniforme
- `reg_lambda` ∈ [1e-4, 10.0] log-uniforme
- `learning_rate` ∈ [0.01, 0.3] log-uniforme

In [22]:
import optuna
from sklearn.metrics import average_precision_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial: optuna.Trial) -> float:
    model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 16, 128),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 50),
        feature_fraction=trial.suggest_float("feature_fraction", 0.5, 1.0),
        bagging_fraction=trial.suggest_float("bagging_fraction", 0.5, 1.0),
        bagging_freq=trial.suggest_int("bagging_freq", 1, 10),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        is_unbalance=True,
        objective="binary",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(
        X_train_lgb, y_train,
        eval_set=[(X_val_lgb, y_val)],
        eval_metric="average_precision",
        callbacks=[early_stopping(50, verbose=False), log_evaluation(0)],
    )
    proba = model.predict_proba(X_val_lgb)[:, 1]
    return average_precision_score(y_val, proba)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nMeilleur AP val : {study.best_value:.4f}")
print("Meilleurs paramètres :")
for k, v in study.best_params.items():
    print(f"  - {k:22s} : {v}")

c:\Users\LauraCarolinaFuentos\Documents\scoring-meelo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Best trial: 24. Best value: 0.331057: 100%|██████████| 30/30 [00:09<00:00,  3.19it/s]


Meilleur AP val : 0.3311
Meilleurs paramètres :
  - learning_rate          : 0.2193085088572082
  - num_leaves             : 39
  - max_depth              : 4
  - min_child_samples      : 15
  - feature_fraction       : 0.7719254961335225
  - bagging_fraction       : 0.8022048670779759
  - bagging_freq           : 5
  - reg_alpha              : 0.010532641273155459
  - reg_lambda             : 0.01207578534501731


In [23]:
# Ré-entraînement du modèle final avec les meilleurs paramètres Optuna
lgb_best = LGBMClassifier(
    **study.best_params,
    n_estimators=1000,
    is_unbalance=True,
    objective="binary",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
lgb_best.fit(
    X_train_lgb, y_train,
    eval_set=[(X_val_lgb, y_val)],
    eval_metric="average_precision",
    callbacks=[early_stopping(50, verbose=False), log_evaluation(0)],
)
print(f"LightGBM-opt entraîné  (best_iteration = {lgb_best.best_iteration_})")

proba_lgbopt_val   = lgb_best.predict_proba(X_val_lgb)[:, 1]
proba_lgbopt_train = lgb_best.predict_proba(X_train_lgb)[:, 1]

lgbopt_result = mdl.evaluate_and_log(
    model_name="LightGBM-opt",
    run_name="d_lightgbm_optuna",
    params={
        "model_type":         "LGBMClassifier",
        "best_iteration":     lgb_best.best_iteration_,
        "optuna_n_trials":    len(study.trials),
        "optuna_best_ap_val": round(float(study.best_value), 4),
        **study.best_params,
        "n_estimators_max":   1000,
        "is_unbalance":       True,
        "encoding":           "native_categorical",
    },
    y_val=y_val,       proba_val=proba_lgbopt_val,
    y_train=y_train,   proba_train=proba_lgbopt_train,
    model=lgb_best,
    model_flavor="lightgbm",
    tags={"stage": "d_optuna", "encoding": "native_cat"},
)

LightGBM-opt entraîné  (best_iteration = 34)
── LightGBM-opt — val ──
  AP (PR-AUC)   : 0.3311
  ROC-AUC       : 0.7768
  Gini          : 0.5536
  Seuil         : 0.788
  Faux Positifs : 8  ← métrique cible
              precision    recall  f1-score   support

        payé      0.956     0.988     0.972       665
      impayé      0.619     0.302     0.406        43

    accuracy                          0.946       708
   macro avg      0.788     0.645     0.689       708
weighted avg      0.936     0.946     0.938       708

── LightGBM-opt — train ──
  AP (PR-AUC)   : 0.7662
  ROC-AUC       : 0.8372
  Gini          : 0.6744
  Seuil         : 0.788
  Faux Positifs : 63  ← métrique cible
              precision    recall  f1-score   support

        payé      0.685     0.969     0.802      2007
      impayé      0.864     0.309     0.455      1297

    accuracy                          0.710      3304
   macro avg      0.774     0.639     0.629      3304
weighted avg      0.755     0

2026/06/30 16:39:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/06/30 16:39:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


### e. Comparatif final des runs MLflow

Récap des 4 runs (a, b, c, d) triés par PR-AUC sur val. La métrique cible
métier reste **`val_fp`** (faux positifs) à seuil contraint, mais on garde
`val_ap` comme score de tri car indépendant du seuil.

Pour explorer interactivement les runs : `mlflow ui` dans le terminal
depuis le dossier du projet, puis ouvrir http://localhost:5000.

In [24]:
comparison = mdl.compare_runs("scoring_impaye", metric_for_sort="val_ap")

# Affichage compact : on ne montre que les colonnes les plus parlantes
display_cols = [c for c in [
    "run_name", "val_ap", "val_gini", "val_fp",
    "val_precision", "val_recall",
    "train_ap", "train_gini", "train_precision",
] if c in comparison.columns]

comparison[display_cols].round(4)


,run_name,val_ap,val_gini,val_fp,val_precision,val_recall,train_ap,train_gini,train_precision
0,d_lightgbm_optuna,0.3311,0.5536,8.0,0.6190,0.3023,0.7662,0.6744,0.8642
1,c_catboost_baseline,0.3092,0.4988,29.0,0.3409,0.3488,0.7420,0.6307,0.7643
2,b_lightgbm_baseline,0.3039,0.5048,26.0,0.3333,0.3023,0.7967,0.7294,0.8287
3,a_random_forest_baseline,0.2597,0.4873,35.0,0.2708,0.3023,0.7610,0.6586,0.7733


## Phase III: Interprétabilité & Analyse des Résultats

### a. SHAP Values - LightGBM-opt (meilleur modèle par AP val)

`shap.TreeExplainer` exploite la structure des arbres pour calculer des valeurs
de Shapley exactes : chaque feature reçoit sa contribution marginale au score
de chaque client de val.

- **Beeswarm** : un point = un client. Axe X = contribution au score (positif → pousse vers impayé).
  La couleur encode la valeur de la feature (bleu = faible, rouge = élevé).
- **Bar plot** : importance globale = moyenne de |SHAP| sur tous les clients de val.

In [25]:
import shap
import plotly.express as px
import mlflow
import mlflow.lightgbm
import mlflow.sklearn
import mlflow.catboost
from mlflow.tracking import MlflowClient
from pandas.api.types import CategoricalDtype

# Recherche du meilleur run par val_precision_unpaid dans MLflow
client = MlflowClient()
experiment = client.get_experiment_by_name("scoring_impaye")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.val_precision_unpaid DESC"],
    max_results=1,
)
best_run = runs[0]
run_name = best_run.data.tags.get("mlflow.runName", best_run.info.run_id)
val_prec = best_run.data.metrics.get("val_precision_unpaid", float("nan"))
print(f"Meilleur run par val_precision : {run_name}  (val_precision = {val_prec:.4f})")

model_uri = f"runs:/{best_run.info.run_id}/model"
model_info = mlflow.models.get_model_info(model_uri)
flavors = set(model_info.flavors.keys())

if "lightgbm" in flavors:
    best_model = mlflow.lightgbm.load_model(model_uri)
elif "catboost" in flavors:
    best_model = mlflow.catboost.load_model(model_uri)
elif "sklearn" in flavors:
    best_model = mlflow.sklearn.load_model(model_uri)
else:
    raise ValueError(f"Flavor non reconnue parmi : {flavors}")

print(f"Modèle chargé  (type : {type(best_model).__name__})")

# Préparation de X_val_shap (visu) et X_shap_input (calcul SHAP)
if "lightgbm" in flavors:
    cat_dtypes = {c: CategoricalDtype(categories=X_train[c].unique()) for c in CATEGORICAL_FEATURES}
    X_val_shap = X_val.astype(cat_dtypes).copy()
    X_shap_coded = X_val_shap.copy()
    for c in CATEGORICAL_FEATURES:
        X_shap_coded[c] = X_shap_coded[c].cat.codes
    X_shap_input = X_shap_coded.to_numpy(dtype=float)
elif "catboost" in flavors:
    X_val_shap = X_val.copy()
    for c in CATEGORICAL_FEATURES:
        X_val_shap[c] = X_val_shap[c].astype(str)
    X_shap_input = X_val_shap
else:
    X_val_shap = X_val.copy()
    X_shap_input = X_val_shap

# Calcul des SHAP values
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_shap_input)

if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

# Beeswarm — top 15 features, un point par client
mean_abs = np.abs(sv).mean(axis=0)
top15_idx = np.argsort(mean_abs)[::-1][:15]

rows = []
for i in top15_idx:
    col = X_val_shap.iloc[:, i]
    try:
        col_vals = col.cat.codes.values.astype(float)
    except AttributeError:
        col_vals = col.values.astype(float)
    vmin, vmax = col_vals.min(), col_vals.max()
    norm = (col_vals - vmin) / (vmax - vmin + 1e-9)
    for shap_val, norm_val in zip(sv[:, i], norm):
        rows.append({"feature": ALL_FEATURES[i], "SHAP": shap_val, "feature_value": norm_val})

df_beeswarm = pd.DataFrame(rows)
feat_order = [ALL_FEATURES[i] for i in top15_idx[::-1]]

SHAP_COLORS = [[0.0, "#3b4cc0"], [0.5, "#f5f5f5"], [1.0, "#b40426"]]

fig = px.scatter(
    df_beeswarm,
    x="SHAP", y="feature",
    color="feature_value",
    color_continuous_scale=SHAP_COLORS,
    title=f"SHAP Beeswarm — {run_name} (meilleur val_precision)",
    labels={"SHAP": "Valeur SHAP", "feature": "", "feature_value": "Valeur feature"},
    category_orders={"feature": feat_order},
    opacity=0.7,
)
fig.add_vline(x=0, line_dash="dash", line_color="grey")
fig.update_layout(width=850, height=520, coloraxis_colorbar=dict(title="Valeur<br>feature"))
fig.show()

Meilleur run par val_precision : d_lightgbm_optuna  (val_precision = 0.6190)
Modèle chargé  (type : LGBMClassifier)


c:\Users\LauraCarolinaFuentos\Documents\scoring-meelo\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [26]:
# Bar plot — importance moyenne |SHAP| par feature (top 15)
mean_abs = np.abs(sv).mean(axis=0)
top15_idx = np.argsort(mean_abs)[::-1][:15]

df_bar = pd.DataFrame({
    "feature":       [ALL_FEATURES[i] for i in top15_idx],
    "mean_abs_SHAP": mean_abs[top15_idx],
}).sort_values("mean_abs_SHAP")

fig = px.bar(
    df_bar,
    x="mean_abs_SHAP", y="feature",
    orientation="h",
    title=f"Importance SHAP — mean |SHAP| — {run_name}",
    labels={"mean_abs_SHAP": "mean |SHAP|", "feature": ""},
    color="mean_abs_SHAP",
    color_continuous_scale="Blues",
)
fig.update_layout(width=700, height=500, coloraxis_showscale=False)
fig.show()

### b. Précision@K et Recall@K

Au lieu d'un seuil fixe, on trie les clients par score décroissant et on
mesure, pour chaque top-K%, combien d'impayés on capture (recall) et à quel
taux de fausses alertes (1 − précision).

Cela permet de répondre à la question métier : *"Si on contacte les X clients
les plus risqués, combien seront de vrais impayés ?"*

In [27]:
import plotly.graph_objects as go
import mlflow
import mlflow.lightgbm
import mlflow.sklearn
import mlflow.catboost
from mlflow.tracking import MlflowClient
from pandas.api.types import CategoricalDtype

# Chargement de tous les runs depuis MLflow
client = MlflowClient()
experiment = client.get_experiment_by_name("scoring_impaye")
all_runs = client.search_runs(experiment_ids=[experiment.experiment_id])
run_map = {r.data.tags.get("mlflow.runName", r.info.run_id): r for r in all_runs}

# Préparation de X_val pour chaque flavor (une seule fois)
_cat_dtypes = {c: CategoricalDtype(categories=X_train[c].unique()) for c in CATEGORICAL_FEATURES}
_X_val_lgb_coded = X_val.astype(_cat_dtypes).copy()
for c in CATEGORICAL_FEATURES:
    _X_val_lgb_coded[c] = _X_val_lgb_coded[c].cat.codes
_X_val_lgb_np = _X_val_lgb_coded.to_numpy(dtype=float)

_X_val_cb = X_val.copy()
for c in CATEGORICAL_FEATURES:
    _X_val_cb[c] = _X_val_cb[c].astype(str)

RUNS_CONFIG = [
    ("d_lightgbm_optuna",        "LightGBM-opt",   "#B279A2"),
    ("b_lightgbm_baseline",      "LightGBM",        "#F58518"),
    ("c_catboost_baseline",      "CatBoost",        "#54A24B"),
    ("a_random_forest_baseline", "Random Forest",   "#4C78A8"),
]

def _get_proba(model_uri, flavors):
    if "lightgbm" in flavors:
        m = mlflow.lightgbm.load_model(model_uri)
        return m.predict(_X_val_lgb_np)          # Booster.predict → prob classe 1
    if "catboost" in flavors:
        m = mlflow.catboost.load_model(model_uri)
        return m.predict_proba(_X_val_cb)[:, 1]
    if "sklearn" in flavors:
        m = mlflow.sklearn.load_model(model_uri)
        return m.predict_proba(X_val)[:, 1]
    raise ValueError(f"Flavor non reconnue : {flavors}")

models_eval = []
for run_name_key, display_name, color in RUNS_CONFIG:
    if run_name_key not in run_map:
        print(f"Run '{run_name_key}' non trouvé dans MLflow, ignoré.")
        continue
    run = run_map[run_name_key]
    uri = f"runs:/{run.info.run_id}/model"
    flavors = set(mlflow.models.get_model_info(uri).flavors.keys())
    proba = _get_proba(uri, flavors)
    models_eval.append((display_name, proba, color))
    print(f"{display_name}")

def precision_recall_at_k(y_true, y_proba):
    n = len(y_true)
    n_pos = int(np.sum(y_true))
    sorted_idx = np.argsort(y_proba)[::-1]
    y_sorted = np.asarray(y_true)[sorted_idx]
    ks = np.arange(1, n + 1)
    cum_tp = np.cumsum(y_sorted)
    return ks / n * 100, cum_tp / ks, cum_tp / n_pos

fig_p = go.Figure()
fig_r = go.Figure()

for name, proba, color in models_eval:
    k_pct, prec_k, rec_k = precision_recall_at_k(y_val, proba)
    kw = dict(mode="lines", line=dict(color=color, width=2))
    fig_p.add_trace(go.Scatter(
        x=k_pct, y=prec_k, name=name,
        hovertemplate="Top %{x:.1f}%<br>Précision : %{y:.3f}<extra>" + name + "</extra>", **kw))
    fig_r.add_trace(go.Scatter(
        x=k_pct, y=rec_k, name=name,
        hovertemplate="Top %{x:.1f}%<br>Recall : %{y:.3f}<extra>" + name + "</extra>", **kw))

baseline = float(y_val.mean())
fig_p.add_hline(y=baseline, line_dash="dash", line_color="grey",
                annotation_text=f"Baseline ({baseline:.2f})",
                annotation_position="bottom right")

layout_common = dict(
    xaxis=dict(title="Top K% de la population (trié par score décroissant)", range=[0, 100]),
    width=750, height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig_p.update_layout(title="Précision@K — val",
                    yaxis=dict(title="Précision", range=[0, 1]), **layout_common)
fig_r.update_layout(title="Recall@K — val",
                    yaxis=dict(title="Recall", range=[0, 1]), **layout_common)
fig_p.show()
fig_r.show()

c:\Users\LauraCarolinaFuentos\Documents\scoring-meelo\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\LauraCarolinaFuentos\Documents\scoring-meelo\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM-opt
LightGBM
CatBoost
Random Forest


In [29]:
K_PCTS = [10, 15, 20, 25, 30]

rows = []
for name, proba, _ in models_eval:
    k_pct_arr, prec_k_arr, rec_k_arr = precision_recall_at_k(y_val, proba)
    row = {"Modèle": name}
    for k in K_PCTS:
        idx = int(np.searchsorted(k_pct_arr, k))
        row[f"Precision@{k}%"] = round(float(prec_k_arr[idx]), 3)
        row[f"Recall@{k}%"]    = round(float(rec_k_arr[idx]),  3)
    rows.append(row)

cols_ordered = [f"{metric}@{k}%" for k in K_PCTS for metric in ("Precision", "Recall")]
df_topk = pd.DataFrame(rows).set_index("Modèle")[cols_ordered].T

prec_rows = [c for c in df_topk.index if c.startswith("Precision")]
rec_rows  = [c for c in df_topk.index if c.startswith("Recall")]

df_topk.style \
    .format("{:.3f}") \
    .background_gradient(subset=pd.IndexSlice[prec_rows, :], cmap="Greens", axis=1) \
    .background_gradient(subset=pd.IndexSlice[rec_rows,  :], cmap="Blues",  axis=1)

Modèle,LightGBM-opt,LightGBM,CatBoost,Random Forest
Precision@10%,0.141,0.085,0.225,0.197
Recall@10%,0.233,0.140,0.372,0.326
Precision@15%,0.206,0.159,0.196,0.178
Recall@15%,0.512,0.395,0.488,0.442
Precision@20%,0.176,0.169,0.176,0.176
Recall@20%,0.581,0.558,0.581,0.581
Precision@25%,0.153,0.136,0.158,0.153
Recall@25%,0.628,0.558,0.651,0.628
Precision@30%,0.136,0.122,0.131,0.127
Recall@30%,0.674,0.605,0.651,0.628


## Conclusion

### Modèle retenu
Quatre modèles ont été comparés sur **val**, seuil calé sous contrainte `recall ≥ 0.30` :

| Modèle | AP val | Gini val | Faux positifs | Précision (impayé) |
|--------|--------|----------|---------------|--------------------|
| **LightGBM-opt (Optuna)** | **0.331** | **0.554** | **8** | **0.62** |
| CatBoost | 0.309 | 0.499 | 29 | 0.34 |
| LightGBM baseline | 0.304 | 0.505 | 26 | 0.33 |
| Random Forest | 0.260 | 0.487 | 35 | 0.27 |

→ **LightGBM-opt est le meilleur sur tous les axes** : AP la plus haute (tri indépendant du seuil) **et** KPI métier (minimiser les faux positifs à recall fixé). Au seuil retenu, il ne déclenche que **8 fausses alertes** pour une **précision de 62 %** sur les impayés. L'optimisation Optuna apporte ici un gain net sur le LightGBM baseline (FP : 26 → 8).

### Enseignements
- **Drivers du risque** : `bank_name` domine largement (IV ≈ 0.76), suivi de `phone_carrier`, `email_domain`, `ported_phone_number` et `urban_density`, cohérent entre l'IV de l'EDA, l'IV post-feature-engineering et les valeurs SHAP.
- **Précision@K** : trier les clients par score et cibler le top-K% capture nettement plus d'impayés qu'un tirage aléatoire → usage opérationnel par priorisation validé.

### Limites
- **Surapprentissage** : écart marqué train/val (AP ~0.74-0.80 en train vs ~0.26-0.33 en val) → le signal disponible reste limité.
- **Évaluation bruitée** : val/test ne comptent que ~6-12 % d'impayés (biais de maturité du split temporel), d'où des métriques sensibles à quelques cas.
- **`bank_name`** : IV « suspect » (> 0.5) → rester vigilant sur une éventuelle fuite avant mise en production.

### Pistes
- **Construire le dataset avec une définition de la cible identique en train/val/test** (fenêtre d'observation fixe, ex. impayé à 6 mois), afin d'éliminer le biais de maturité et de rendre les taux d'impayé comparables entre splits.

- Enrichir les données (historique de paiement, comportement) pour réduire le sous-apprentissage du signal.
- Re-calibrer le seuil selon le coût réel FP/FN côté métier.
- Surveiller le **time-drift** en production via le suivi MLflow déjà en place.
